In [1]:
# Install required libraries if missing in your environment
!pip install dash pandas plotly numpy

import dash
from dash import html, dcc
from dash.dependencies import Input, Output
from dash import dash_table
import pandas as pd
import plotly.graph_objects as go
import numpy as np

print("Libraries imported successfully!")

   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   ------ --------------------------------- 1.6/9.3 MB 8.4 MB/s eta 0:00:01
   --------------- ------------------------ 3.7/9.3 MB 9.6 MB/s eta 0:00:01
   ------------------------- -------------- 6.0/9.3 MB 10.1 MB/s eta 0:00:01
   ----------------------------------- ---- 8.1/9.3 MB 10.3 MB/s eta 0:00:01
   ---------------------------------------- 9.3/9.3 MB 10.2 MB/s  0:00:00

   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   -------------------------- ------------- 2/3 [dash]
   ---------------------------------------- 3/3 [dash]


In [2]:
csv_content = """year,film,genre,box_office_millions,imdb_rating,total_wins
2026,Anora,Comedy,35.4,7.9,3
2025,Sinners,Thriller,215.0,8.2,5
2024,Oppenheimer,Biography,957.2,8.9,7
2023,Everything Everywhere All at Once,Sci-Fi,143.4,7.8,7
2022,CODA,Drama,2.1,8.0,3
2021,Nomadland,Drama,3.9,7.3,3
2020,Parasite,Thriller,263.1,8.5,4
2019,Green Book,Biography,321.8,8.2,3
2018,The Shape of Water,Romance,195.9,7.3,4
2017,Moonlight,Drama,65.3,7.4,3
2016,Spotlight,Biography,98.7,8.1,2
2015,Birdman,Comedy,103.2,7.7,4
2014,12 Years a Slave,Biography,187.7,8.1,3
2013,Argo,Thriller,232.3,7.7,3
2012,The Artist,Comedy,133.4,7.9,5
2011,The King's Speech,Biography,414.2,8.0,4
2010,The Hurt Locker,Action,49.2,7.5,6
2009,Slumdog Millionaire,Drama,378.1,8.0,8
2008,No Country for Old Men,Crime,171.6,8.2,4
2007,The Departed,Crime,291.5,8.5,4
2006,Crash,Drama,101.3,7.7,3
2005,Million Dollar Baby,Drama,216.7,8.1,4
2004,The Lord of the Rings: The Return of the King,Action,1146.0,9.0,11
2003,Chicago,Musical,306.8,7.2,6
2002,A Beautiful Mind,Biography,313.5,8.2,4
2001,Gladiator,Action,460.5,8.5,5
2000,American Beauty,Drama,356.3,8.3,5"""

with open("oscars.csv", "w", encoding="utf-8") as f:
    f.write(csv_content)

# Verify dataset loading
df = pd.read_csv("oscars.csv")
print("Dataset loaded successfully! Total films:", len(df))
display(df.head())

Dataset loaded successfully! Total films: 27


,year,film,genre,box_office_millions,imdb_rating,total_wins
0,2026,Anora,Comedy,35.4,7.9,3
1,2025,Sinners,Thriller,215.0,8.2,5
2,2024,Oppenheimer,Biography,957.2,8.9,7
3,2023,Everything Everywhere All at Once,Sci-Fi,143.4,7.8,7
4,2022,CODA,Drama,2.1,8.0,3


In [3]:
# Load dataset
df = pd.read_csv("oscars.csv")
df.columns = df.columns.str.strip()

# Initialize Dash app
app = dash.Dash(__name__, suppress_callback_exceptions=True)

# App Layout
app.layout = html.Div([
    html.H1("🎬 Oscars Box Office & Critical Acclaimed Intelligence", style={"textAlign": "center", "color": "#B8860B"}),
    html.P("An interactive dashboard exploring box office earnings, IMDb ratings, and Academy Awards success for legendary films.", style={"textAlign": "center", "fontStyle": "italic"}),

    dcc.Tabs(id="mainTabs", value="overviewTab", children=[
        dcc.Tab(label="Overview", value="overviewTab"),
        dcc.Tab(label="Box Office vs Ratings", value="graphicsTab"),
        dcc.Tab(label="Data Table", value="tableTab"),
        dcc.Tab(label="Summary Statistics", value="summaryTab"),
    ]),

    html.Hr(),

    # Filters Section
    html.Div([
        html.Div([
            html.Label("Select Genre:"),
            dcc.Dropdown(
                id="genreFilter",
                options=[{"label": "All", "value": "All"}] +
                        [{"label": g, "value": g} for g in sorted(df["genre"].unique())],
                value=["All"],
                multi=True,
                style={"width": "95%"}
            )
        ], style={'width': '35%', 'display': 'inline-block'}),

        html.Div([
            html.Label("Minimum Box Office ($M):"),
            dcc.Slider(
                id="boxOfficeFilter",
                min=float(df["box_office_millions"].min()),
                max=float(df["box_office_millions"].max()),
                value=float(df["box_office_millions"].min()),
                marks={0: '0M', 300: '300M', 600: '600M', 1000: '1000M'},
                tooltip={"placement": "bottom", "always_visible": True}
            )
        ], style={'width': '60%', 'display': 'inline-block', 'paddingLeft': '5%'})
    ], style={"padding": "15px", "backgroundColor": "#f9f9f9", "borderRadius": "5px"}),

    html.Hr(),

    html.Div(id="tabContent"),
    dcc.Store(id="filteredData")
])

# Callback for Data Filtering
@app.callback(
    Output("filteredData", "data"),
    [Input("genreFilter", "value"),
     Input("boxOfficeFilter", "value")]
)
def filter_data(genresel, minbox):
    dff = df.copy()
    if "All" not in genresel:
        dff = dff[dff["genre"].isin(genresel)]
    dff = dff[dff["box_office_millions"] >= minbox]
    return dff.to_dict("records")

# Callback for Main Tabs Rendering
@app.callback(
    Output("tabContent", "children"),
    [Input("mainTabs", "value"),
     Input("filteredData", "data")]
)
def render_tabs(tab, data):
    if not data:
        return html.Div("Loading data...")
    dff = pd.DataFrame(data)

    if tab == "overviewTab":
        fig = go.Figure(go.Histogram(x=dff["total_wins"], marker_color="#B8860B"))
        fig.update_layout(title="Distribution of Total Oscar Wins (Real Masterpieces)", xaxis_title="Wins", yaxis_title="Count")
        return html.Div([
            html.H3("Overview of Academy Award Winning Masterpieces"),
            html.P("This dashboard analyzes iconic Best Picture winners, evaluating their commercial box office earnings against public IMDb feedback."),
            dcc.Graph(figure=fig)
        ])

    if tab == "graphicsTab":
        return html.Div([
            dcc.Tabs(id="nestedTabs", value="scatter", children=[
                dcc.Tab(label="Box Office vs IMDb Rating", value="scatter"),
                dcc.Tab(label="Average Wins by Genre", value="bar"),
                dcc.Tab(label="IMDb Rating Distribution", value="box")
            ]),
            html.Div(id="nestedContent", style={"paddingTop": "20px"})
        ])

    if tab == "tableTab":
        return html.Div([
            html.H3("Filtered Data Table (6 Columns - Real Films)"),
            dash_table.DataTable(
                columns=[{"name": col, "id": col} for col in dff.columns],
                data=dff.to_dict("records"),
                page_size=12,
                style_table={"overflowX": "auto"},
                style_cell={"textAlign": "center"},
                filter_action="native",
                sort_action="native"
            )
        ])

    if tab == "summaryTab":
        return html.Div([
            html.H3("Summary Statistics"),
            html.Pre(dff.describe().to_string(), style={"background": "#f4f4f4", "padding": "15px", "borderRadius": "5px"})
        ])

# Callback for Nested Graphics Tabs
@app.callback(
    Output("nestedContent", "children"),
    [Input("nestedTabs", "value"),
     Input("filteredData", "data")]
)
def nested_graphs(nestedTab, data):
    if not data:
        return html.Div("Waiting for data...")
    dff = pd.DataFrame(data)

    if nestedTab == "scatter":
        fig = go.Figure(data=go.Scatter(
            x=dff["box_office_millions"],
            y=dff["imdb_rating"],
            mode="markers+text",
            text=dff["film"],
            textposition="top center",
            marker=dict(size=12, color=dff["total_wins"], colorscale="Sunset", showscale=True, colorbar=dict(title="Total Wins"))
        ))
        fig.update_layout(title="Box Office vs IMDb Rating", xaxis_title="Box Office ($M)", yaxis_title="IMDb Rating", height=600)
        return dcc.Graph(figure=fig)

    if nestedTab == "bar":
        if len(dff) > 0:
            avg_wins = dff.groupby("genre")["total_wins"].mean().reset_index()
            fig = go.Figure(go.Bar(x=avg_wins["genre"], y=avg_wins["total_wins"], marker_color="#DAA520"))
            fig.update_layout(title="Average Total Wins by Genre", xaxis_title="Genre", yaxis_title="Average Wins")
        else:
            fig = go.Figure()
        return dcc.Graph(figure=fig)

    if nestedTab == "box":
        fig = go.Figure(data=go.Box(y=dff["imdb_rating"], name="IMDb Rating", marker_color="#B8860B"))
        fig.update_layout(title="Distribution of IMDb Ratings")
        return dcc.Graph(figure=fig)

# Run local server inside Jupyter
app.run(debug=True, port=8050)